# Components of the PIC sweep (sweep_components.dag, sweep_components_palumbo.dag)

In [ ]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sunsim import observations as obs, local_path

BASE = local_path('components_results')          # ${BASE}/components of PIC (or its folder sets_<TAG>)
TAG = 'components_m45s_size_4096_N60'
START = '2015-07-18'                             # t = 0 of the simulations
PARTS = ('spot', 'plage', 'network', 'spot_plage', 'spot_plage_network')
TOTALS = ('spot_plage', 'spot_plage_network')
MAX_FF_FC, MAX_FF_SP = 20.0, 5.0                 # epochs left out (bad HMI images), as obs.sweep_stats
COLOURS = {'spot': 'k', 'plage': 'C3', 'network': 'C1', 'spot_plage': 'C0', 'spot_plage_network': 'C2'}

SETS = next(f for f in (os.path.join(str(BASE), 'sets_' + TAG), str(BASE)) if os.path.isdir(os.path.join(f, 'parts'))
            or any(os.path.isdir(os.path.join(f, b, 'spot')) for b in os.listdir(f)))
BISECTORS = sorted(b for b in os.listdir(SETS) if os.path.isdir(os.path.join(SETS, b, 'spot')))
NAMES = sorted({os.path.basename(f) for b in BISECTORS for f in glob.glob(os.path.join(SETS, b, 'spot', '*'))
                if os.path.isdir(f)})
print(SETS)
print('bisectors:', BISECTORS)
print('sets:', *NAMES, sep='\n  ')

In [ ]:
def series(bisector, part, name):
    """Time series of one part (obs._read_series), the bad epochs of the total spot_plage_network left out."""
    s = obs._read_series(os.path.join(SETS, bisector, part, name))
    ref = obs._read_series(os.path.join(SETS, bisector, 'spot_plage_network', name))
    bad = set(ref['time'][(ref['ff_fc'] > MAX_FF_FC) | (ref['ff_sp'] > MAX_FF_SP)])
    keep = ~np.isin(s['time'], list(bad))
    return {k: v[keep] for k, v in s.items()}


def short(name):
    return name.replace('_' + TAG, '')

## Statistics of the totals

In [ ]:
rows = []
for b in BISECTORS:
    for part in TOTALS:
        for f in sorted(glob.glob(os.path.join(SETS, b, part, '*', 'stats.json'))):
            rows.append({'bisector': b, 'part': part, 'set': short(os.path.basename(os.path.dirname(f))),
                         **json.load(open(f))})
df = pd.DataFrame(rows)
cols = ['bisector', 'part', 'set', 'rv_rms', 'rv_mad', 'fwhm_rms', 'bis_rms', 'sorce_rms', 'tsis_rms', 'rv_obs_rms']
df[[c for c in cols if c in df]].sort_values(['part', 'set', 'rv_rms'])

In [ ]:
for part in TOTALS:
    d = df[df['part'] == part].pivot(index='set', columns='bisector', values='rv_rms')
    ax = d.plot.bar(figsize=(8, 4), rot=0)
    ax.set_ylabel('RV rms vs HARPS-N [m/s]')
    ax.set_title(part)
    ax.set_ylim(0.9 * np.nanmin(d.values), 1.02 * np.nanmax(d.values))
    plt.show()

## RV of the parts

In [ ]:
def rv_std(b, part, name):
    return np.nanstd(series(b, part, name)['rv'])

for name in NAMES:
    print(short(name))
    display(pd.DataFrame({b: {p: rv_std(b, p, name) for p in PARTS} for b in BISECTORS}).round(3))

In [ ]:
for name in NAMES:
    fig, axes = plt.subplots(len(BISECTORS), 1, figsize=(15, 2.8 * len(BISECTORS)), sharex=True, sharey=True,
                             squeeze=False)
    for ax, b in zip(axes[:, 0], BISECTORS):
        for p in PARTS:
            s = series(b, p, name)
            ax.plot(s['time'], s['rv'] - np.nanmedian(s['rv']), '.', ms=1.5, c=COLOURS[p], label=p)
        ax.set_ylabel('RV [m/s]')
        ax.set_title(b, loc='left', fontsize=10)
    axes[0, 0].legend(markerscale=6, ncol=len(PARTS), fontsize=8)
    axes[-1, 0].set_xlabel('Time [days since %s]' % START)
    fig.suptitle(short(name))
    fig.tight_layout()
    plt.show()

## RV of the spots for every bisector

In [ ]:
for name in NAMES:
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={'width_ratios': [3, 1]})
    for i, b in enumerate(BISECTORS):
        s = series(b, 'spot', name)
        rv = s['rv'] - np.nanmedian(s['rv'])
        ax.plot(s['time'], rv, '.', ms=1.5, c='C%d' % i, label=b)
        ax2.plot(s['ff_sp'], rv, '.', ms=1.5, c='C%d' % i)
    ax.set_xlabel('Time [days since %s]' % START)
    ax.set_ylabel('RV of the spots [m/s]')
    ax.legend(markerscale=6)
    ax2.set_xlabel('ff_sp [%]')
    fig.suptitle(short(name))
    fig.tight_layout()
    plt.show()

## FWHM and BIS of the parts

In [ ]:
for name in NAMES:
    for q in ('fwhm', 'bis'):
        fig, axes = plt.subplots(len(BISECTORS), 1, figsize=(15, 2.5 * len(BISECTORS)), sharex=True, sharey=True,
                                 squeeze=False)
        for ax, b in zip(axes[:, 0], BISECTORS):
            for p in PARTS:
                s = series(b, p, name)
                if q in s:
                    ax.plot(s['time'], s[q] - np.nanmedian(s[q]), '.', ms=1.5, c=COLOURS[p], label=p)
            ax.set_ylabel('%s [m/s]' % q.upper())
            ax.set_title(b, loc='left', fontsize=10)
        axes[0, 0].legend(markerscale=6, ncol=len(PARTS), fontsize=8)
        axes[-1, 0].set_xlabel('Time [days since %s]' % START)
        fig.suptitle('%s: %s' % (short(name), q.upper()))
        fig.tight_layout()
        plt.show()

## Light curve of the parts

In [ ]:
for name in NAMES:
    fig, ax = plt.subplots(figsize=(15, 4))
    for p in PARTS:
        s = series(BISECTORS[0], p, name)                 # the same for every bisector
        ax.plot(s['time'], 1e6 * (s['lc'] / np.nanmean(s['lc']) - 1), '.', ms=1.5, c=COLOURS[p], label=p)
    ax.set_xlabel('Time [days since %s]' % START)
    ax.set_ylabel('Flux [ppm]')
    ax.legend(markerscale=6, ncol=len(PARTS))
    ax.set_title(short(name))
    plt.show()

## Against HARPS-N and TSI

In [ ]:
harps = obs.load_harpsn(local_path('harpsn_sun'), START)
sorce, tsis = obs.load_sorce(local_path('sorce_tsi'), START), obs.load_tsis(local_path('tsis_tsi'), START)
cuts = obs.load_interventions(local_path('harpsn_interventions'), START)

NAME = NAMES[0]                                  # the set compared
PART = 'spot_plage_network'                      # one of TOTALS
for b in BISECTORS:
    s = series(b, PART, NAME)
    rv = [obs.compare_rv(harps, s['time'], s[q], quantity=q, interventions=cuts) for q in ('rv', 'fwhm', 'bis')]
    ph = [obs.compare_photometry(tsi, s['time'], s['lc']) for tsi in (sorce, tsis)]
    print('%s, %s, %s' % (b, PART, short(NAME)))
    obs.print_summary(ph, rv)
    fig, axes = obs.plot_comparison(ph, rv[0], start=START, figsize=(15, 7))
    fig.suptitle('%s: %s' % (b, PART))
    plt.show()

## RMS maps and minima (sweep_components_rawccf.dag)

The stats.json of the totals of a run, as maps of I_th (rows) against A_th (columns), one panel per T_sp; the RV for
every bisector (one row of panels each), the TSI once (it does not depend on the bisector). The star marks the minimum
of every panel, the white box the minimum of the whole figure. RUN = '_rawccf': the run with ccf_norm = 2 (raw ring
CCFs, settings_components.sh); RUN = '': the runs before 2026-10-27 (every ring CCF normalised).

In [ ]:
RUN = '_rawccf'
SETS_RUN = SETS + RUN if os.path.basename(SETS) == 'sets_' + TAG else os.path.join(SETS, '..', 'sets_' + TAG + RUN)
BISECTORS_RUN = sorted(b for b in os.listdir(SETS_RUN) if os.path.isdir(os.path.join(SETS_RUN, b, 'spot_plage')))
rows = []
for b in BISECTORS_RUN:
    for part in TOTALS:
        for f in sorted(glob.glob(os.path.join(SETS_RUN, b, part, '*', 'stats.json'))):
            rows.append({'bisector': b, 'part': part, 'set': short(os.path.basename(os.path.dirname(f))),
                         **json.load(open(f))})
stats = pd.DataFrame(rows)
print(SETS_RUN)
print('bisectors:', BISECTORS_RUN)
print(len(stats), 'sets with stats.json (expected %d)' % (len(BISECTORS_RUN) * len(TOTALS) * len(NAMES)))


In [ ]:
QUANTITIES = {'rv_rms': ('HARPS-N RV', 'm/s', '%.3f'), 'sorce_rms': ('SORCE/TIM TSI', 'ppm', '%.0f'),
              'tsis_rms': ('TSIS-1/TIM TSI', 'ppm', '%.0f')}


def rms_maps(stats, value, part, bisectors=None):
    """Maps of value (a column of stats) for one part of TOTALS: rows = bisectors, columns = T_sp, I_th against A_th."""
    title, unit, fmt = QUANTITIES[value]
    d = stats[(stats['part'] == part) & stats[value].notna()]
    bisectors = bisectors or sorted(d['bisector'].unique())
    temps, i_s, a_s = sorted(d['T_sp'].unique()), sorted(d['I_th'].unique()), sorted(d['A_th'].unique())
    vmin, vmax = d[d['bisector'].isin(bisectors)][value].min(), d[d['bisector'].isin(bisectors)][value].max()
    fig, axes = plt.subplots(len(bisectors), len(temps), figsize=(3.2 * len(temps) + 1, 2.6 * len(bisectors) + 0.6),
                             sharex=True, sharey=True, squeeze=False)
    best = d[d['bisector'].isin(bisectors)].sort_values(value).iloc[0]
    for r, b in enumerate(bisectors):
        for c, T in enumerate(temps):
            ax = axes[r, c]
            sub = d[(d['bisector'] == b) & (d['T_sp'] == T)]
            grid = sub.pivot_table(index='I_th', columns='A_th', values=value).reindex(index=i_s, columns=a_s).values
            im = ax.imshow(grid, origin='lower', aspect='auto', cmap='inferno', vmin=vmin, vmax=vmax)
            for (y, x), v in np.ndenumerate(grid):
                if np.isfinite(v):
                    ax.text(x, y, fmt % v, ha='center', va='center', fontsize=8,
                            color='w' if v < vmin + 0.6 * (vmax - vmin) else 'k')
            if np.isfinite(grid).any():
                y, x = np.unravel_index(np.nanargmin(grid), grid.shape)
                ax.plot(x, y + 0.3, '*', c='C2', ms=9)
                if b == best['bisector'] and T == best['T_sp']:
                    ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1, fill=False, ec='w', lw=2.5))
            ax.set_xticks(range(len(a_s)), ['%g' % a for a in a_s])
            ax.set_yticks(range(len(i_s)), ['%g' % i for i in i_s])
            if r == 0:
                ax.set_title('T_sp = %g K' % T)
            if r == len(bisectors) - 1:
                ax.set_xlabel('A_th [uHem]')
            if c == 0:
                ax.set_ylabel('%s\nI_th' % b if len(bisectors) > 1 else 'I_th')
    fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02, label='%s rms of the residuals [%s]' % (title, unit))
    fig.suptitle('%s: %s%s' % (title, part, RUN))
    plt.show()
    return best


for part in TOTALS:
    rms_maps(stats, 'rv_rms', part)
    for value in ('sorce_rms', 'tsis_rms'):
        rms_maps(stats, value, part, bisectors=BISECTORS_RUN[:1])      # the TSI is the same for every bisector


### Where the minima are

The set with the smallest rms of every observation (and, for HARPS-N, the best set of every bisector), with the rms of
the other observations for the same set, and the joint score of component_fit.joint_score (profile likelihood, the
variance of every data set fitted; TSI residuals correlated over 27 days): smaller is better, a few units is not
significant.

In [ ]:
COLS = ['part', 'bisector', 'I_th', 'A_th', 'T_sp', 'rv_rms', 'sorce_rms', 'tsis_rms', 'fwhm_rms', 'bis_rms']
minima = pd.concat([stats.sort_values(v).groupby('part').head(1).assign(minimum_of=v) for v in QUANTITIES])
display(minima[['minimum_of'] + COLS].sort_values(['part', 'minimum_of']).round(3))

print('Best set of every bisector for HARPS-N RV:')
display(stats.sort_values('rv_rms').groupby(['part', 'bisector']).head(1)[COLS].sort_values(['part', 'rv_rms']).round(3))

# joint RV + TSIS-1 score (the same as sunsim/rv_tsi_reconcile/component_fit.py of the project files)
tau_rv, tau_tsi = 1.0, 27.0
stats['joint'] = (stats['rv_n'] / tau_rv * np.log(stats['rv_rms'] ** 2)
                  + stats['tsis_n'] / tau_tsi * np.log(stats['tsis_rms'] ** 2))
stats['joint'] -= stats['joint'].min()
print('Best sets for RV and TSIS-1 together:')
display(stats.sort_values('joint')[COLS + ['joint']].head(10).round(3))
